# NTGEN: General Nanotube Generation from the Alexandria 1D Database

Generate new general (multi-element) nanotube candidates with the NTGEN diffusion
model, seeded by **real 1D nanotube structures** from the Alexandria database
(SCIGEN-style mask-constrained denoising, "Pathway 3" — no retraining required).

**Pipeline**: sample a real nanotube template from
`data/alx_1D/nanotube_templates.npz` (2,216 structures distilled from the 7,002
ASE structures in `alexandria_1d_nanotubes.pkl`) → `SC_DBTemplate` pins the
template's atoms (per-atom species + fractional coords + cell) as the known
skeleton via `mask_x`/`mask_l`/`mask_t` → `sample_scigen` reverse diffusion
re-imposes the pinned atoms at every step while the model denoises the remaining
decorating atoms → `pymatgen` → CIF.

**Constraint keys** (`sc_dict` in `script/sc_utils.py`):
`'alx'` real Alexandria template (this notebook), `'ntb'` parametric skeleton
tube, `'cnt'` carbon nanotube, `'van'` unconstrained.

> **Dataset note**: run `'alx'` with a *general* dataset (`mp_20`). `carbon_24`
> would force every atom to carbon and flatten the template's real species.

## Running on Google Colab

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Cell 1 auto-detects Colab and clones the repo; Cell 2 installs the extra
   dependencies (torch + CUDA come preinstalled on Colab).
3. **Prerequisite**: `models/NTGEN-edit/` — including the 648 KB
   `data/alx_1D/nanotube_templates.npz` cache — must be **pushed to the GitHub
   repo** (it is untracked by default; `git add models/ && git push` locally
   first). Alternatively, upload the tree to Google Drive and use Option B in
   Cell 1.

In [ ]:
# --- Environment & paths (Google Colab or local) -------------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Option A (default): clone the GitHub repo. models/NTGEN-edit must be
    # committed & pushed for this to work (see the Colab note above).
    REPO_URL = 'https://github.com/3venthatguy/NTU-IQM.git'
    REPO_DIR = Path('/content/NTU-IQM')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
                       check=True)

    # Option B: code on Google Drive instead of GitHub — uncomment:
    # from google.colab import drive; drive.mount('/content/gdrive')
    # REPO_DIR = Path('/content/gdrive/MyDrive/NTU-IQM')

    PROJECT_DIR = (REPO_DIR / 'models' / 'NTGEN-edit').resolve()
    NOTEBOOK_DIR = (REPO_DIR / 'models' / 'NTGEN_generation').resolve()
else:
    NOTEBOOK_DIR = Path.cwd().resolve()          # capture before chdir below
    PROJECT_DIR = Path(os.environ.get(
        'NTGEN_PROJECT_DIR', NOTEBOOK_DIR.parent / 'NTGEN-edit')).resolve()

assert PROJECT_DIR.exists(), (
    f'NTGEN-edit not found at {PROJECT_DIR}. On Colab: push models/ to GitHub '
    'first, or mount Drive and point REPO_DIR at your copy (Option B above).')

# The package dir is named ntgent/ but all imports & hydra targets say scigen.*
# -> self-heal with a symlink if needed.
if not (PROJECT_DIR / 'scigen').exists() and (PROJECT_DIR / 'ntgent').exists():
    os.symlink('ntgent', PROJECT_DIR / 'scigen')
    print('created symlink scigen -> ntgent')

os.environ.setdefault('PROJECT_ROOT', str(PROJECT_DIR))
os.environ.setdefault('HYDRA_JOBS', str(PROJECT_DIR))
os.environ.setdefault('WANDB_DIR', str(PROJECT_DIR / 'wandb'))
os.environ.setdefault('WANDB_MODE', 'disabled')   # no wandb login prompts
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

for p in (str(PROJECT_DIR), str(PROJECT_DIR / 'script')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_DIR)   # gen_utils opens ./data/... relative to cwd
print('Colab       :', IN_COLAB)
print('PROJECT_DIR :', PROJECT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)

In [ ]:
# --- Dependencies (Colab: torch + CUDA come preinstalled) ----------------------
if IN_COLAB:
    import torch
    tv = torch.__version__.split('+')[0]
    cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
    !pip install -q hydra-core omegaconf pytorch-lightning pymatgen torch_geometric
    # torch_scatter needs the wheel matching Colab's torch/CUDA build:
    !pip install -q torch_scatter -f https://data.pyg.org/whl/torch-{tv}+{cu}.html
    import hydra, pymatgen, torch_geometric, torch_scatter
    print('deps OK | torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
else:
    print('local run: using the current environment as-is')

In [ ]:
# --- Alexandria 1D template cache ----------------------------------------------
# gen_utils mmaps data/alx_1D/nanotube_templates.npz (compact, no ase needed).
# Build it once from the raw ASE pickle if it is missing.
import subprocess

alx_dir = PROJECT_DIR / 'data' / 'alx_1D'
cache = alx_dir / 'nanotube_templates.npz'
if not cache.exists():
    raw = alx_dir / 'alexandria_1d_nanotubes.pkl'
    assert raw.exists(), (
        f'{cache.name} is missing and the raw pickle {raw.name} is not here either. '
        'On Colab: commit the 648 KB nanotube_templates.npz to the repo (recommended) '
        'or upload the 48 MB pickle to data/alx_1D/ and rerun this cell.')
    print('template cache missing -> building from', raw.name, '...')
    subprocess.run([sys.executable, str(alx_dir / 'build_templates.py')], check=True)
print('template cache:', cache, f'({cache.stat().st_size/1024:.0f} KB)')

In [ ]:
# --- Diffusion checkpoint (public SCIGEN mp_20 weights from Figshare) ----------
import json, zipfile
from urllib.request import urlopen, urlretrieve

MODEL_PATH = PROJECT_DIR / 'models' / 'mp_20'

if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print('downloading', f['name'], '...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()
print('checkpoints:', [c.name for c in MODEL_PATH.glob('*.ckpt')])

In [ ]:
# --- Load model & bind the SCIGEN sampler --------------------------------------
import torch
import hydra
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

def load_model_for_inference(model_path, device='cpu'):
    model_path = Path(model_path)
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')
    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'), ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))
    model = model.to(device)
    model.eval()
    return model, cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model_for_inference(MODEL_PATH, device=device)

from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)
print('model ready on', device)

## Generation settings

- `SC_TYPE='alx'`: each candidate starts from one **real Alexandria 1D nanotube**
  drawn uniformly among templates whose atom count fits `natm_range` (with at
  least one slot left for a model-placed decorating atom). The template's
  per-atom species are pinned exactly — multi-element skeletons stay
  multi-element. If no template fits the range, that draw silently falls back to
  the parametric `SC_Nanotube` ('ntb').
- `DATASET='mp_20'` supplies the atom-count distribution (up to 20 atoms) and
  keeps atom types free for the diffusion model — required for multi-element
  templates.
- `KNOWN_SPECIES` only feeds the bond-length sampler; `SC_DBTemplate` ignores it
  (the template carries its own species/geometry). Any element in the metallic
  KDE works.
- `reduced_mask=False` is required: template atoms are pinned per-coordinate,
  `mask_x (N,3)`.

In [ ]:
SC_TYPE = 'alx'        # 'alx' Alexandria template | 'ntb' parametric | 'cnt' | 'van'
DATASET = 'mp_20'      # general dataset: preserves multi-element template species
KNOWN_SPECIES = ['Fe'] # bond-len sampler only; ignored by SC_DBTemplate
BATCH_SIZE = 4
NUM_BATCHES = 1
FRAC_Z = 0.5           # unused by 'alx' (template sets coords); kept for 'ntb'
STEP_LR = 5e-6
SEED = 42

SC_NATM_RANGE = {'alx': [1, 20], 'ntb': [4, 20], 'cnt': [1, 20], 'van': [1, 20]}
natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])
total_structures = BATCH_SIZE * NUM_BATCHES
print(f'{SC_TYPE}: {total_structures} structures, natm_range {natm_range}')

In [ ]:
# --- Build the constrained dataset & run reverse diffusion ---------------------
from tqdm.auto import tqdm
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset
from sc_utils import chemical_symbols

test_set = SampleDataset(
    dataset=DATASET,
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=KNOWN_SPECIES,
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)

# Peek at the drawn templates: pinned skeleton size + species per candidate.
for i in range(len(test_set)):
    d = test_set[i]
    K = int(d.num_known)
    species = sorted({chemical_symbols[int(z)] for z in d.atom_types_known[:K]})
    print(f'  [{i}] pinned {K} atoms {species} -> total {int(d.num_atoms[0])}')

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

all_frac_coords, all_atom_types, all_lattices = [], [], []
all_num_atoms, all_num_known = [], []
for batch in tqdm(test_loader, desc='Generating structures'):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)
    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)
print('generated:', num_atoms.tolist(), 'atoms per structure (pinned skeleton:', num_known.tolist(), ')')

In [ ]:
# --- Convert to pymatgen structures --------------------------------------------
import numpy as np
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure

def lattices_to_params(lat):
    """(3,3) lattice matrix -> (lengths[3], angles_deg[3])."""
    lengths = np.linalg.norm(lat, axis=1)
    angles = np.zeros(3)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos = np.dot(lat[j], lat[k]) / (lengths[j] * lengths[k])
        angles[i] = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    return lengths, angles

structures = []
start = 0
for i in range(num_atoms.shape[0]):
    n_i = int(num_atoms[i])
    coords_i = frac_coords[start:start + n_i].numpy()
    types_i = atom_types[start:start + n_i].numpy()
    start += n_i
    lengths_i, angles_i = lattices_to_params(lattices[i].numpy())
    species = [chemical_symbols[int(t)] for t in types_i]
    try:
        structure = Structure(
            Lattice.from_parameters(*lengths_i, *angles_i),
            species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
    except Exception as e:
        structures.append(None)
        print(f'structure {i}: conversion failed ({e})')
print(sum(s is not None for s in structures), '/', len(structures), 'structures converted')

In [ ]:
# --- Quick 3D look at one candidate --------------------------------------------
import matplotlib.pyplot as plt

s0 = next((s for s in structures if s is not None), None)
if s0 is not None:
    xyz = s0.cart_coords
    zs = [site.specie.Z for site in s0]
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=zs, cmap='viridis', s=60)
    ax.set_title(s0.composition.reduced_formula)
    plt.show()

In [ ]:
# --- Export CIF files ----------------------------------------------------------
output_dir = NOTEBOOK_DIR / 'generated_cifs'
output_dir.mkdir(parents=True, exist_ok=True)
n_written = 0
for i, structure in enumerate(structures):
    if structure is None:
        continue
    formula = structure.composition.reduced_formula
    cif_path = output_dir / f'{SC_TYPE}_{formula}_{i:03d}.cif'
    structure.to(filename=str(cif_path), fmt='cif')
    n_written += 1
print(f'wrote {n_written} CIFs to', output_dir)

if IN_COLAB and n_written:
    # Colab storage is wiped when the runtime ends -> zip & download the CIFs.
    # (To save to Drive instead: mount it and copy output_dir there.)
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive('/content/generated_cifs', 'zip', output_dir)
    files.download(zip_path)

## Notes

- **Template pool**: the cache keeps Alexandria structures with ≤ 64 atoms
  (2,216 of 7,002); with `natm_range=[1, 20]` about 370 templates are drawable.
  Raise `MAX_NATM` in `data/alx_1D/build_templates.py` and rebuild to widen the
  pool (the mp_20 checkpoint was trained on ≤ 20 atoms, so larger cells are
  out-of-distribution).
- **Fallback**: draws with no fitting template use the parametric `SC_Nanotube`
  — check the per-candidate printout above to see what was actually pinned.
- **Other modes**: set `SC_TYPE='ntb'` for parametric skeleton tubes (single
  element from `KNOWN_SPECIES`), or use `05_ctgen_generation.ipynb` in this
  folder for carbon nanotubes (`'cnt'` + `carbon_24`).